# Local SPECTER Cache Validation

This notebook validates the existing `specter_text_cache.pt` locally. It does not need `text_registry.jsonl`; alignment is checked directly against `train.jsonl`, `val.jsonl`, and `test.jsonl` by using each row's `positive_texts[0].text`, which is also how Stage 3 training indexes the cache.

It answers two separate questions:

1. What is inside the `.pt` file, and are the vectors already unit-normalized, corrupted, duplicated, or misaligned?
2. Does the Stage 3 training code normalize raw cached SPECTER vectors before the text projection, or only normalize projected embeddings inside InfoNCE/evaluation?


In [ ]:
from __future__ import annotations

from pathlib import Path
import ast
import csv
import hashlib
import json
import os
import random
import re
import sys
import time
from collections import Counter
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", Path.cwd())).expanduser()
if not (REPO_DIR / "experiments/3dcnn").exists():
    REPO_DIR = Path.cwd()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

CACHE_PATH = Path(os.environ.get(
    "NEUROVLM_TEXT_EMBEDDING_CACHE",
    "experiments/3dcnn/atlas_free_cnn/cache/text_embeddings/specter_text_cache.pt",
)).expanduser()
SPLIT_DIR = Path(os.environ.get(
    "NEUROVLM_UNIFIED_SPLIT_DIR",
    "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
)).expanduser()
OUT_DIR = Path(os.environ.get(
    "NEUROVLM_SPECTER_VALIDATION_OUT",
    f"specter_cache_validation_{time.strftime('%Y%m%d_%H%M%S')}",
)).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_DIR)
print("Cache:", CACHE_PATH, "exists=", CACHE_PATH.exists())
print("Split dir:", SPLIT_DIR, "exists=", SPLIT_DIR.exists())
print("Output:", OUT_DIR)


In [ ]:
def write_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(payload, f, indent=2, default=str)

def write_csv(path: str | Path, rows: list[dict[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if not rows:
        path.write_text("")
        return
    keys = sorted({k for row in rows for k in row})
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    with Path(path).open() as f:
        return [json.loads(line) for line in f if line.strip()]

def primary_text(row: dict[str, Any]) -> dict[str, str]:
    positives = row.get("positive_texts", []) or []
    if not positives:
        return {"text": "", "text_id": ""}
    first = positives[0]
    return {"text": str(first.get("text", "")), "text_id": str(first.get("text_id") or first.get("id") or first.get("text") or "")}

def domain_from_source(value: str) -> str:
    source = str(value or "").lower()
    if source == "pubmed" or source.startswith("pubmed"):
        return "pubmed"
    if source == "nilearn" or source.startswith("nilearn"):
        return "nilearn"
    if source == "neurovault" or source.startswith("neurovault"):
        return "neurovault"
    return source


In [ ]:
# Load and inspect the .pt structure first. For the current HF/local cache this is:
# dict[exact_text_string] -> torch.float32 tensor, shape [768]
payload = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
structure = {"payload_type": type(payload).__name__}

if isinstance(payload, dict):
    keys = list(payload.keys())
    values = list(payload.values())
    structure.update({
        "num_entries": len(payload),
        "sample_key_reprs": [repr(k)[:240] for k in keys[:5]],
        "sample_value_types": [type(v).__name__ for v in values[:5]],
        "sample_value_shapes": [list(v.shape) if torch.is_tensor(v) else None for v in values[:5]],
        "sample_value_dtypes": [str(v.dtype) if torch.is_tensor(v) else None for v in values[:5]],
    })
else:
    raise TypeError(f"Unsupported cache payload type: {type(payload)}")

if not all(isinstance(k, str) for k in keys):
    raise TypeError("Expected cache keys to be exact text strings")
if not all(torch.is_tensor(v) for v in values):
    raise TypeError("Expected cache values to be tensors")

dims = Counter(int(v.numel()) for v in values)
dtypes = Counter(str(v.dtype) for v in values)
structure.update({"dimension_counts": dict(dims), "dtype_counts": dict(dtypes)})
write_json(OUT_DIR / "specter_cache_structure.json", structure)
structure


In [ ]:
# Numeric validation: norms, NaNs/Infs, duplicate vectors, and pairwise cosine on a reproducible subset.
texts = list(payload.keys())
emb = torch.stack([payload[t].float().flatten() for t in texts])
norms = emb.norm(dim=1)

rng = np.random.default_rng(0)
subset_n = min(int(os.environ.get("NEUROVLM_PAIRWISE_SUBSET_N", "2048")), emb.shape[0])
subset_idx = rng.choice(emb.shape[0], size=subset_n, replace=False)
sub = F.normalize(emb[subset_idx], dim=1, eps=1e-8)
pairwise = sub @ sub.T
pairwise_mask = ~torch.eye(subset_n, dtype=torch.bool)

hashes = [hashlib.sha256(emb[i].numpy().tobytes()).hexdigest() for i in range(emb.shape[0])]
summary = {
    "cache_path": str(CACHE_PATH),
    "n_vectors": int(emb.shape[0]),
    "embedding_dim": int(emb.shape[1]),
    "norm_mean": float(norms.mean().item()),
    "norm_std": float(norms.std().item()),
    "norm_min": float(norms.min().item()),
    "norm_median": float(norms.median().item()),
    "norm_max": float(norms.max().item()),
    "fraction_norm_within_1e_4_of_1": float((norms.sub(1).abs() <= 1e-4).float().mean().item()),
    "fraction_norm_within_1e_3_of_1": float((norms.sub(1).abs() <= 1e-3).float().mean().item()),
    "nan_count": int(torch.isnan(emb).sum().item()),
    "inf_count": int(torch.isinf(emb).sum().item()),
    "duplicate_vector_count": int(len(hashes) - len(set(hashes))),
    "fraction_duplicate_vectors": float(1.0 - len(set(hashes)) / len(hashes)),
    "per_dimension_mean_mean": float(emb.mean(dim=0).mean().item()),
    "per_dimension_mean_abs_mean": float(emb.mean(dim=0).abs().mean().item()),
    "mean_pairwise_cosine_subset": float(pairwise[pairwise_mask].mean().item()),
    "subset_n_for_pairwise_cosine": int(subset_n),
}
write_json(OUT_DIR / "specter_cache_numeric_audit.json", summary)
pd.DataFrame([summary])


In [ ]:
# Alignment validation: no text_registry.jsonl needed.
# Stage 3 training indexes cache by exact positive_texts[0].text, so we check the same thing.
alignment_rows = []
map_rows = []
for split in ["train", "val", "test"]:
    path = SPLIT_DIR / f"{split}.jsonl"
    if not path.exists():
        alignment_rows.append({"split": split, "status": "missing_split", "path": str(path)})
        continue
    rows = read_jsonl(path)
    for domain in ["pubmed", "nilearn", "neurovault"]:
        drows = [r for r in rows if domain_from_source(r.get("source", "")) == domain]
        missing = []
        text_ids = []
        primary_texts = []
        for row in drows:
            pt = primary_text(row)
            text_ids.append(pt["text_id"])
            primary_texts.append(pt["text"])
            if pt["text"] not in payload:
                missing.append({"map_id": row.get("map_id", ""), "text_id": pt["text_id"], "text_preview": pt["text"][:160]})
            map_rows.append({"split": split, "domain": domain, "map_id": row.get("map_id", ""), "source": row.get("source", ""), "primary_text_id": pt["text_id"], "cache_key_found": pt["text"] in payload, "primary_text": pt["text"]})
        alignment_rows.append({
            "split": split,
            "domain": domain,
            "n_rows": len(drows),
            "unique_primary_texts": len(set(primary_texts)),
            "unique_text_ids": len(set(text_ids)),
            "missing_cache_key_count": len(missing),
            "missing_examples_json": json.dumps(missing[:10]),
            "status": "ok" if not missing else "missing_cache_keys",
        })
write_csv(OUT_DIR / "specter_cache_alignment_audit.csv", alignment_rows)
write_csv(OUT_DIR / "map_to_cache_key_alignment.csv", map_rows)
pd.DataFrame(alignment_rows)


In [ ]:
# Training-code audit: does Stage 3 normalize raw cached SPECTER before projection?
# Expected answer from current code: no. It loads cache vectors, returns them as batch["text"],
# applies text_proj(text), and InfoNCE normalizes the projected embeddings.
FILES_TO_AUDIT = {
    "stage3_trainer": REPO_DIR / "experiments/3dcnn/train_ale_cnn.py",
    "loss": REPO_DIR / "src/neurovlm/loss.py",
}
PATTERNS = {
    "cache_loader": "def load_text_embedding_cache",
    "dataset_returns_cache_text": '"text": self.text_cache[text].float()',
    "forward_applies_text_proj": "text_emb = self.text_proj(text)",
    "infonce_normalizes_projected_inputs": "image = F.normalize(image, dim=1)",
}

def snippet_around(path: Path, needle: str, context: int = 5) -> dict[str, Any]:
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines):
        if needle in line:
            lo = max(0, i - context)
            hi = min(len(lines), i + context + 1)
            return {"path": str(path), "line": i + 1, "needle": needle, "snippet": "\n".join(f"{j+1}: {lines[j]}" for j in range(lo, hi))}
    return {"path": str(path), "line": None, "needle": needle, "snippet": "NOT FOUND"}

audit = {
    "conclusion": "Stage 3 does not L2-normalize raw cached SPECTER vectors before text_proj. Raw cache vectors are fed directly into text_proj; InfoNCE normalizes projected text and brain embeddings inside the loss.",
    "implication": "If the cache vectors are already unit-normalized, that happened during cache creation, not in Stage 3 training. If they are not unit-normalized, Stage 3 still trained with those raw vectors as text_proj input.",
    "snippets": {
        name: snippet_around(FILES_TO_AUDIT["loss"] if "infonce" in name else FILES_TO_AUDIT["stage3_trainer"], needle)
        for name, needle in PATTERNS.items()
    },
}
write_json(OUT_DIR / "stage3_text_embedding_normalization_code_audit.json", audit)
for name, item in audit["snippets"].items():
    print(f"\n## {name}\n{item['snippet']}")
print("\nCONCLUSION:", audit["conclusion"])


In [ ]:
# Optional numerical reproduction. Off by default because it needs model weights/dependencies and may download from HF.
# It compares cached vectors against raw SPECTER2, unit(raw), raw-empty, and unit(raw-empty).
RUN_REPRODUCTION = os.environ.get("NEUROVLM_RUN_SPECTER_REPRODUCTION", "0") == "1"
REPRO_SAMPLE_N = int(os.environ.get("NEUROVLM_SPECTER_REPRO_SAMPLE_N", "16"))
repro_rows = []
repro_summary = {"status": "not_run", "reason": "Set NEUROVLM_RUN_SPECTER_REPRODUCTION=1 to re-encode a sample."}

if RUN_REPRODUCTION:
    from atlas_free_cnn.training.model_wrappers import encode_texts_specter
    rng = random.Random(0)
    sample_texts = list(texts)
    rng.shuffle(sample_texts)
    sample_texts = sample_texts[:REPRO_SAMPLE_N]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    raw = encode_texts_specter(sample_texts + [""], device=device, batch_size=int(os.environ.get("NEUROVLM_SPECTER_BATCH", "8")))
    empty = raw[-1].cpu()
    raw = raw[:-1].cpu()
    candidates = {
        "raw": raw,
        "unit_raw": F.normalize(raw, dim=1, eps=1e-8),
        "raw_minus_empty": raw - empty,
        "unit_raw_minus_empty": F.normalize(raw - empty, dim=1, eps=1e-8),
    }
    for i, text in enumerate(sample_texts):
        cached = payload[text].float().cpu()
        for transform_name, candidate in candidates.items():
            diff = candidate[i] - cached
            repro_rows.append({
                "sample_index": i,
                "transform": transform_name,
                "cosine": float(F.cosine_similarity(candidate[i], cached, dim=0, eps=1e-8).item()),
                "max_abs_diff": float(diff.abs().max().item()),
                "mse": float(diff.pow(2).mean().item()),
                "text_preview": text[:120],
            })
    frame = pd.DataFrame(repro_rows)
    mean_by_transform = frame.groupby("transform")["mse"].mean().sort_values().to_dict()
    repro_summary = {
        "status": "run",
        "sample_n": len(sample_texts),
        "best_transform_by_mean_mse": next(iter(mean_by_transform)),
        "mean_mse_by_transform": mean_by_transform,
        "empty_string_embedding_checksum": hashlib.sha256(empty.numpy().tobytes()).hexdigest(),
    }

write_csv(OUT_DIR / "specter_cache_reproduction_sample.csv", repro_rows)
write_json(OUT_DIR / "specter_cache_reproduction_summary.json", repro_summary)
repro_summary


In [ ]:
# Final local validation summary.
# Important: unit norm is an observed property of the cache file. The code audit above tells where normalization happens during training.
final = {
    "cache_structure": structure,
    "numeric_audit": summary,
    "alignment_status_counts": dict(Counter(row["status"] for row in alignment_rows)),
    "stage3_training_raw_cache_normalized_before_text_proj": False,
    "stage3_training_projected_embeddings_normalized_in_infonce": True,
    "text_registry_required_for_this_validation": False,
    "outputs": str(OUT_DIR),
}
write_json(OUT_DIR / "FINAL_SPECTER_CACHE_VALIDATION_SUMMARY.json", final)
print(json.dumps(final, indent=2, default=str)[:4000])
